# Aula 5.2 - Alura - versao aprimorada\n

In [ ]:

import re
from datetime import datetime

dbutils.widgets.text("gravar", "false")
dbutils.widgets.text("catalogo", "workspace")
dbutils.widgets.text("esquema", "default")

gravar = dbutils.widgets.get("gravar").strip().lower() == "true"
cat = dbutils.widgets.get("catalogo").strip()
esq = dbutils.widgets.get("esquema").strip()

for identificador in [cat, esq]:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z_0-9]*", identificador):
        raise ValueError("Nome de catalogo ou esquema invalido")

dbutils.widgets.text("data_inicio", "20260101")
dbutils.widgets.text("data_fim", "20260110")

dbutils.widgets.text(
    "features",
    "renda_mensal_k,tempo_medio_clique_segundos"
)
dbutils.widgets.text("target", "comprou_eletronico")
import numpy as np
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType, TimestampType
from scipy.stats import ks_2samp
from datetime import datetime



# Novos parâmetros para tornar o código genérico
p_inicio = dbutils.widgets.get("data_inicio")
p_fim = dbutils.widgets.get("data_fim")
p_feature_str = dbutils.widgets.get("features")
p_target = dbutils.widgets.get("target")

# p_inicio = "20260101"
# p_fim = "20260110"

# p_feature_str = "renda_mensal_k,tempo_medio_clique_segundos"
# p_target = "comprou_eletronico"

# Processando a string de features para uma lista Python
lista_features = [f.strip() for f in p_feature_str.split(",") if f.strip()]

print(f"Iniciando Job de Monitoramento. Avaliando partições entre {p_inicio} e {p_fim}")
print(f"Monitorando Features: {lista_features}")
print(f"Monitorando Target: {p_target}\n")

nome_tabela_treino = f"{cat}.{esq}.base_treinamento_modelo_aprimorado"
nome_tabela_prod = f"{cat}.{esq}.base_tabela_prod_aprimorada"
nome_tabela_auditoria = f"{cat}.{esq}.log_monitoramento_drift_aprimorado"

# leitura dos dados de treino
df_treino = spark.table(nome_tabela_treino)

# Pushdown filter aproveitando a partição dia_prtc
df_prod = spark.table(nome_tabela_prod).filter(
    (F.col("dia_prtc") >= p_inicio) & (F.col("dia_prtc") < p_fim)
)


if datetime.strptime(p_inicio, "%Y%m%d") >= datetime.strptime(
    p_fim, "%Y%m%d"
):
    raise ValueError("Periodo invalido")

if not lista_features:
    raise ValueError("Informe as features")

for coluna in lista_features + [p_target]:
    if coluna not in df_treino.columns or coluna not in df_prod.columns:
        raise ValueError("Coluna nao encontrada: " + coluna)

if df_prod.limit(1).count() == 0:
    raise ValueError("Sem dados produtivos no periodo")

logs_execucao = []

def registrar(nome, coluna, valor, limite):
    if "p-value" in nome.lower():
        alerta = float(valor) < limite
    else:
        alerta = float(valor) > limite

    logs_execucao.append((
        datetime.now(),
        p_inicio,
        p_fim,
        nome,
        coluna,
        float(valor),
        float(limite),
        bool(alerta)
    ))

def amostrar(df, coluna):
    base = df.select(coluna).where(F.col(coluna).isNotNull())

    serie = base.sample(
        fraction=0.1, seed=42
    ).limit(5000).toPandas()[coluna]

    if len(serie) < 2:
        serie = base.limit(5000).toPandas()[coluna]

    return serie.to_numpy(dtype=float)

for feature in lista_features:
    arr_treino = amostrar(df_treino, feature)
    arr_prod = amostrar(df_prod, feature)

    if min(len(arr_treino), len(arr_prod)) < 2:
        raise ValueError("Amostra insuficiente: " + feature)

    estatistica, pvalue = ks_2samp(
        arr_treino, arr_prod
    )

    print(
        f"KS {feature}: "
        f"estatistica={estatistica:.4f}, "
        f"p-value={pvalue:.6g}"
    )

    registrar(
        "Feature drift KS: p-value",
        feature,
        pvalue,
        0.05
    )

media_t = df_treino.select(F.avg(p_target)).first()[0]
media_p = df_prod.select(F.avg(p_target)).first()[0]

if media_t is None or media_p is None:
    raise ValueError("Target sem dados validos")

if media_t > 0:
    variacao = abs(media_p - media_t) / media_t
    nome = "Target drift: variacao relativa"
else:
    variacao = abs(media_p - media_t)
    nome = "Target drift: diferenca absoluta"

print(
    f"Target: treino={media_t:.4f}, "
    f"producao={media_p:.4f}, "
    f"variacao={variacao:.4f}"
)

registrar(nome, p_target, variacao, 0.15)

schema = StructType([
    StructField("timestamp_execucao", TimestampType(), False),
    StructField("periodo_inicio", StringType(), False),
    StructField("periodo_fim", StringType(), False),
    StructField("nome_metrica", StringType(), False),
    StructField("variavel", StringType(), False),
    StructField("valor_calculado", DoubleType(), False),
    StructField("threshold_alerta", DoubleType(), False),
    StructField("drift_detectado", BooleanType(), False)
])

df_logs = spark.createDataFrame(
    logs_execucao, schema=schema
)

df_logs.show(truncate=False)

print("TOTAL DE METRICAS:", len(logs_execucao))

if gravar:
    df_logs.write.format("delta").mode("append").saveAsTable(
        nome_tabela_auditoria
    )
    print("LOGS GRAVADOS:", nome_tabela_auditoria)
else:
    print("PREVIA: monitoramento nao gravado")
